# 314 — Investigation Agent

Demonstrates entity network exploration using Claude-generated Cypher
and the `detect_graph_anomalies` FastMCP tool.

**Real scenarios:**
- `BRW-0001` — Patrick Nelson, individual, risk_rating=high
- `ACC-0596` — receiving structured transactions from 6 accounts
- `BRW-0582` — top of a 3-level ownership chain

In [1]:
%run 311_agent_setup.ipynb

/opt/anaconda3/envs/graphrag-finserv/lib/python3.11/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Environment loaded.


2026-03-09 15:36:40,508 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


Connected to Neo4j.

Node counts in graph:
  {'labels': {'CommercialSecured': 2, 'Federal': 1, 'Address': 609, 'Residential': 582, 'Corporate': 31, 'Assessment': 5, 'Collateral': 466, 'ResidentialSecured': 464, 'Industry': 14, 'National': 5, 'PersonalTransaction': 610, 'Threshold': 157, 'CorporateCurrent': 6, 'Borrower': 628, 'BusinessTransaction': 175, 'SpecialAdminRegion': 1, 'BankAccount': 791, 'Chunk': 205, 'Section': 118, 'Jurisdiction': 7, 'Commercial': 27, 'Requirement': 194, 'Transaction': 173, 'LoanApplication': 466, 'Individual': 597, 'Finding': 22, 'Regulation': 3, 'ReasoningStep': 31, 'Officer': 19}}
  ✓  Borrower: 628
  ✓  LoanApplication: 466
  ✓  BankAccount: 791
  ✓  Transaction: 173
  ✓  Regulation: 3
  ✓  Section: 118
  ✓  Requirement: 194
  ✓  Threshold: 157
  ✓  Chunk: 205
Anthropic client ready. Model: claude-sonnet-4-6
OpenAI client ready.
Tool registry: 2 Neo4j MCP + 5 FastMCP = 7 total
  read-neo4j-cypher
  write-neo4j-cypher
  traverse_compliance_path
  retriev

In [2]:
from src.agent.investigation_agent import InvestigationAgent

agent = InvestigationAgent(tools=TOOLS, execute_tool_fn=execute_tool)

## Example 1: High-risk borrower network — BRW-0001

In [3]:
result = agent.run('Show suspicious connections around BRW-0001. Check all anomaly patterns.')
print('Entity:', result.entity_id)
print('\nRisk signals:')
for s in result.risk_signals:
    print(f'  {s}')
print('\nCypher used:', len(result.cypher_used), 'queries')

2026-03-09 15:36:43,662 [INFO] src.agent.investigation_agent: InvestigationAgent: Show suspicious connections around BRW-0001. Check all anomaly patterns.
2026-03-09 15:36:50,908 [INFO] src.agent.investigation_agent: Tool: read-neo4j-cypher(['query'])
2026-03-09 15:36:50,910 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 15:36:51,011 [INFO] src.agent.investigation_agent: Tool: detect_graph_anomalies(['pattern_name', 'entity_id'])
2026-03-09 15:36:51,012 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name', 'entity_id']
2026-03-09 15:36:51,227 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 15:36:51,254 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-09 15:36:51,257 [INFO] src.agent.investigation_agent: Tool: detect_graph_anomalies(['pattern_name', 'entity_id'])
2026-03-09 15:36:51,257 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name', 'entity_i

Entity: BRW-0001

Risk signals:
  [HIGH] Pre-existing HIGH risk rating** — BRW-0001 (Patrick Nelson) carries a `risk_rating: high` in the system with a below-average credit score of **632**. No industry or officer associations are recorded, making income source verification difficult.
  [HIGH] Transaction Structuring Network — 19 flagged transactions across 3 accounts** — The `transaction_structuring` anomaly pattern detected **3 structuring clusters** totalling **AUD $176,185** flowing into accounts held by second-degree connections of BRW-0001:
  [HIGH] Co-jurisdiction clustering with structuring targets** — Graph traversal confirms BRW-0001 shares the **same Australian Federal jurisdiction node** as BRW-0602 (Hassan Martin, also `risk_rating: high`), BRW-0603 (Carlo Scott), and BRW-0604 (Benjamin Wagner) — all three being recipients of the structured funds. This co-residency link is a meaningful network signal.
  [HIGH] BRW-0602 (Hassan Martin) is also HIGH risk and has an active lo

## Example 2: Transaction structuring investigation — ACC-0596

In [4]:
result2 = agent.run(
    'Investigate ACC-0596. Multiple suspicious transactions are flowing into this account. '
    'Who owns it? Where do the funds come from? Is this structuring?'
)
print('Risk signals:')
for s in result2.risk_signals:
    print(f'  {s}')
print('\nConnections:', result2.connections)

2026-03-09 15:38:08,880 [INFO] src.agent.investigation_agent: InvestigationAgent: Investigate ACC-0596. Multiple suspicious transactions are flowing into this account. Who owns it? Where do the funds come from? Is this structuring?
2026-03-09 15:38:16,556 [INFO] src.agent.investigation_agent: Tool: read-neo4j-cypher(['query'])
2026-03-09 15:38:16,558 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 15:38:16,746 [INFO] src.agent.investigation_agent: Tool: read-neo4j-cypher(['query'])
2026-03-09 15:38:16,746 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 15:38:16,792 [INFO] src.agent.investigation_agent: Tool: detect_graph_anomalies(['pattern_name', 'entity_id'])
2026-03-09 15:38:16,792 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name', 'entity_id']
2026-03-09 15:38:17,017 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 15:38:17,042 [INFO] src.graph.connec

Risk signals:
  [HIGH] Classic Structuring Pattern — AUSTRAC Threshold Evasion
  [HIGH] Source of Funds Completely Obscured
  [HIGH] Coordinated Structuring Network — 3 Accounts, $176,185 Total
  [HIGH] Unexplained Wealth — Zero Income, $109K Monthly Balance
  [MEDIUM] Active Loan Under Review — Potential Proceeds of Crime Risk
  [MEDIUM] Pre-existing HIGH Risk Rating

Connections: []


## Example 3: Ownership chain — BRW-0582

In [5]:
result3 = agent.run(
    'Show the full ownership structure behind BRW-0582. '
    'How many levels deep does the chain go? Are there any loans associated with subsidiaries?'
)
print('Risk signals:')
for s in result3.risk_signals:
    print(f'  {s}')
print('\nCypher queries:', len(result3.cypher_used))
for q in result3.cypher_used:
    print(f'  {q[:120]}')

2026-03-09 15:39:51,958 [INFO] src.agent.investigation_agent: InvestigationAgent: Show the full ownership structure behind BRW-0582. How many levels deep does the chain go? Are there any loans associated with subsidiaries?
2026-03-09 15:39:59,556 [INFO] src.agent.investigation_agent: Tool: read-neo4j-cypher(['query'])
2026-03-09 15:39:59,559 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 15:39:59,757 [INFO] src.agent.investigation_agent: Tool: read-neo4j-cypher(['query'])
2026-03-09 15:39:59,759 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 15:39:59,802 [INFO] src.agent.investigation_agent: Tool: detect_graph_anomalies(['pattern_name', 'entity_id'])
2026-03-09 15:39:59,803 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name', 'entity_id']
2026-03-09 15:40:00,000 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 15:40:00,038 [INFO] src.graph.connection: Neo

Risk signals:
  [HIGH] Myanmar-Resident Ultimate Beneficial Owner (UBO)** — Sakura Campbell's registered address is 7 Central Avenue, Yangon, Myanmar (JUR-MM = HIGH AML risk). Critically, no formal `RESIDES_IN` jurisdiction edge exists in the graph, creating a **KYC data gap** that may be masking the true risk classification.
  [HIGH] 3-Level Layered Ownership with Cross-Border Intermediate** — The chain routes through a Hong Kong foreign company (Boronia Asia Ltd) before reaching two Australian Pty Ltd entities. This is a textbook structure for obscuring beneficial ownership: Myanmar individual → HK shell → AU holding → AU operating company.
  [HIGH] Level-3 Subsidiary Guaranteeing a High-LVR Loan** — Atlas Meridian Pty Ltd (BRW-0579), the deepest subsidiary, is the **guarantor on LOAN-0440** ($1,000,000 AUD, LVR = **90%**, status: under_review). The ultimate controller of this guarantor is a Myanmar-resident individual. This creates undisclosed cross-border credit exposure.
  [MEDIUM

## Example 4: Manual review trigger — combined question

In [6]:
result4 = agent.run(
    'Why might LOAN-0002 require manual review? '
    'Check both the borrower and the guarantor if any.'
)
print('Risk signals:')
for s in result4.risk_signals:
    print(f'  {s}')

2026-03-09 15:41:38,525 [INFO] src.agent.investigation_agent: InvestigationAgent: Why might LOAN-0002 require manual review? Check both the borrower and the guarantor if any.
2026-03-09 15:41:43,111 [INFO] src.agent.investigation_agent: Tool: read-neo4j-cypher(['query'])
2026-03-09 15:41:43,115 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 15:41:43,294 [INFO] src.agent.investigation_agent: Tool: detect_graph_anomalies(['pattern_name'])
2026-03-09 15:41:43,294 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name']
2026-03-09 15:41:43,498 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 15:41:43,543 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-09 15:41:43,543 [INFO] src.agent.investigation_agent: Tool: traverse_compliance_path(['entity_id', 'entity_type'])
2026-03-09 15:41:43,544 [INFO] execute_tool: Tool: traverse_compliance_path | inputs: ['entity_id', 'entity_type']


Risk signals:
  [HIGH] — LVR of 95% breaches the APG-223 / APS-112 senior review threshold
  [HIGH] — Borrower income recorded as $0 — critical serviceability gap
  [MEDIUM] — Serviceability buffer compliance unverifiable
  [MEDIUM] — No guarantor; no credit support beyond collateral
  [LOW] — Valuation independence to be confirmed
